In [1]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re


In [17]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_colwidth', None)


In [3]:
df = pd.read_parquet('../../data/final_without_pi_centum_data_with_medical_data_20250411_225729.parquet')

In [4]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_duration', '약_medication_type',
       '약_frequency', '약_duration', '약_compliance', '장치_device_type',
       '장치_usage_pattern', '장치_duration', '장치_compliance', '습관_habit_type',
       '습관_frequency', '습관_awareness', '습관_improvement', '찜질_status',
       '찜질_frequency', '찜질_duration', '찜질_method', '마사지, 스트레칭_type',
       '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       '

### 데이터 전처리

In [5]:
## NaN 처리 

df = df.replace('', np.nan)
df = df.replace('-', np.nan)
df = df.replace('- -', np.nan)
df = df.replace('n/s', np.nan)

df['약_medication_type'] = df['약_medication_type'].replace('없음',np.nan)
df['약_medication_type'] = df['약_medication_type'].replace('약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용함','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('저녁약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다양한 약물','약물 종류 미상')

df['Noise_Code'] = df['Noise_Code'].apply(lambda x : "No-Noise" if x == 0 else "Click" if x == 1 else "Popping" if x == 2 else "Crepitus" if x == 3 else "Unknown")

text_cols = [
    'CC_location','CC_pain_type','CC_painUncomp_desc_jaw','CC_disable_desc_jaw','CC_muscle_joint_desc_stress',
    'CC_dentalHistory_desc','CC_clinic_history_desc','CC_factor_habbit','CC_treat_plan',
    '약_medication_type','약_compliance','장치_device_type','습관_habit_type','습관_awareness'
    ]
numeric_cols = [
    'CC_duration','CC_severity', 'CC_vas', 'CMO_before','CMO_after','MMO_before','MMO_after','Midline_Shift_Amount','CRCO_Amount','Next_Visit_Days'
    ,'Rt_before','Rt_after','Lt_before','Lt_after','Tongue_ridging_Intensity','장치_duration','찜질_duration','마사지, 스트레칭_duration','약_duration'
    ,'M.pal_Pain_Intensity','Cap.pal_Pain_Intensity','Noise_Intensity','Occlusion_lt_Intensity','Occlusion_rt_Intensity', 'oj','ob'
    ]
category_cols = [
    '장치_usage_pattern','장치_compliance', '습관_frequency', '습관_improvement','약_frequency',
    '찜질_status','찜질_frequency', '마사지, 스트레칭_frequency','마사지, 스트레칭_method' ,'deviation_pattern_type','deviation_direction',
    'Cap.pal_Pain_Direction','Cap.pal_Pain_Situation','M.pal_Pain_Direction','M.pal_Pain_Situation','Noise_Code','Noise_Direction',
    'Noise_Situation','Occlusion_lt_number','Occlusion_rt_number','Midline_Shift_Jaw','Midline_Shift_Direction_x','Midline_Shift_Direction_y',
    'CRCO_Direction_x','CRCO_Direction_y','Cap.pal_Pain_Intensity','deviation_intensity',
    '마사지, 스트레칭_frequency','마사지, 스트레칭_type','찜질_method'
    ]


In [6]:
## Vas 추출 

def get_vas_sentence(text):
    if isinstance(text, str) and 'vas' in text.lower():
        vas_index = text.lower().find('vas')
        vas_after = text[vas_index:]
        vas_after = vas_after.replace(" ", "")
        vas_after = vas_after.lower()
        return vas_after

def get_vas_value(text):    
    pattern = re.compile(
        # r"(?i)VAS(?:\s*(?:,|~|->)?\s*(\d+(?:\.\d+)?))+"
        r"(?i)VAS(?:\s*(?:[,\.\~\/]|->)\s*)?"      # 'VAS' 뒤에 선택적으로 구분자가 올 수 있음
        r"(\d+(?:\.\d+)?)"                         # 첫 번째 숫자 (정수 또는 소수)
        r"(?:\s*(?:[,\.\~\/]|->)\s*(\d+(?:\.\d+)?))*"  # 이후 반복되는 구분자와 숫자; 마지막 반복의 캡처 그룹에 최종 숫자가 남음
        )
    if text:
        match = pattern.search(text)
        if match:
            # print(match.group(1))
            return match.group(2) if match.group(2) is not None else match.group(1)
            # return match.group(1)
    else:
        return None


df['CC_vas_sentence'] = df['CC'].apply(get_vas_sentence)
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('-->','->')
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('--->','->')
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('---->','->')

df['CC_vas'] = df['CC_vas_sentence'].apply(get_vas_value)

df.drop(columns=['CC_vas_sentence'], inplace=True)

In [7]:
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
    '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method', 'CMO_before', 'CMO_after', 'MMO_before',
       'MMO_after', 'deviation_pattern_type', 'deviation_direction',
       'deviation_intensity', 'Cap.pal_Pain_Intensity',
       'Cap.pal_Pain_Direction', 'Cap.pal_Pain_Situation',
       'M.pal_Pain_Intensity', 'M.pal_Pain_Direction', 'M.pal_Pain_Situation',
       'Noise_Code', 'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days', 'CC_vas'
    ]]


In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28162 entries, 0 to 28161
Data columns (total 80 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   환자번호                         28162 non-null  object        
 1   날짜                           28162 non-null  datetime64[ns]
 2   CC                           28161 non-null  object        
 3   약                            5897 non-null   object        
 4   장치                           11187 non-null  object        
 5   습관                           16035 non-null  object        
 6   찜질                           16917 non-null  object        
 7   마사지, 스트레칭                    5935 non-null   object        
 8   PI                           17480 non-null  object        
 9   치료계획                         20489 non-null  object        
 10  End feel                     25630 non-null  object        
 11  CC_location                  16251 non-nu

In [9]:
def structure_patient_data(row):
    structured_text = ""
    for column in text_columns:
        if pd.notna(row[column]) and row[column]:
            structured_text += f"{column}: {row[column]}\n"
    return structured_text

In [16]:
df.columns


Index(['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', '치료계획',
       'End feel', 'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method', 'CMO_before', 'CMO_after', 'MMO_before',
       'MMO_after', 'deviation_pattern_type', 'deviation_direction',
       'deviation_intensity', 'Cap.pal_Pain_Intensity',
       'Cap.pal_Pain_Direction', 'Cap.pal_Pain_Situation',
       'M.pal_Pain_Intensity', 'M.

### 모델

#### 임베딩을 위한 맥락 부여

In [46]:
import pandas as pd
import openai
import numpy as np

def create_contextual_text(row):
    """
    환자 데이터를 임상적 맥락을 강화한 방식으로 구조화하여 텍스트 생성
    """
    # 최종 임상 텍스트를 담을 섹션별 컨테이너
    clinical_sections = []
    
    # 1. 주호소 및 증상 섹션 (Core Symptoms)
    symptoms_context = []
    
    # 증상 위치와 종류 통합
    if pd.notna(row['CC_location']) and pd.notna(row['CC_pain_type']):
        symptoms_context.append(f"주호소: 환자는 {row['CC_location']}에 {row['CC_pain_type']}을 호소합니다.")
    elif pd.notna(row['CC_location']):
        symptoms_context.append(f"주호소 위치: {row['CC_location']}")
    elif pd.notna(row['CC_pain_type']):
        symptoms_context.append(f"통증 유형: {row['CC_pain_type']}")
    
    # 턱 관련 통증과 불편감 세부 정보 추가
    if pd.notna(row['CC_painUncomp_desc_jaw']):
        symptoms_context.append(f"턱 관련 통증 및 불편감: {row['CC_painUncomp_desc_jaw']}")
    
    # 기능적 제한 정보 추가
    if pd.notna(row['CC_disable_desc_jaw']):
        symptoms_context.append(f"턱 기능 제한: {row['CC_disable_desc_jaw']}")
    
    # 근육과 관절 관련 증상 추가
    if pd.notna(row['CC_muscle_joint_desc_stress']):
        symptoms_context.append(f"근육 및 관절 상태: {row['CC_muscle_joint_desc_stress']}")
    
    # 통증 강도와 지속 기간 정보
    if pd.notna(row['CC_severity']):
        symptoms_context.append(f"통증 강도(1-5): {row['CC_severity']}")
    
    if pd.notna(row['CC_vas']):
        symptoms_context.append(f"VAS 통증 점수: {row['CC_vas']}")
    
    if pd.notna(row['CC_duration']):
        symptoms_context.append(f"증상 지속 기간: {row['CC_duration']}")
    
    # 증상 섹션을 통합하여 전체 맥락에 추가
    if symptoms_context:
        clinical_sections.append("【증상 정보】\n" + "\n".join(symptoms_context))
    
    # 2. 병력 및 습관 섹션 (History & Habits)
    history_context = []
    
    # 치과 관련 과거력
    if pd.notna(row['CC_dentalHistory_desc']):
        history_context.append(f"치과 병력: {row['CC_dentalHistory_desc']}")
    
    # 클리닉 방문 이력
    if pd.notna(row['CC_clinic_history_desc']):
        history_context.append(f"턱관절 관련 과거 치료: {row['CC_clinic_history_desc']}")
    
    # 생활 습관 요인
    if pd.notna(row['CC_factor_habbit']):
        history_context.append(f"생활 습관 요인: {row['CC_factor_habbit']}")
    
    # 습관 관련 상세 정보
    habit_details = []
    if pd.notna(row['습관_habit_type']):
        habit_details.append(f"습관 유형: {row['습관_habit_type']}")
    
    if pd.notna(row['습관_frequency']):
        habit_details.append(f"습관 빈도: {row['습관_frequency']}")
    
    if pd.notna(row['습관_awareness']):
        habit_details.append(f"습관 인지 여부: {row['습관_awareness']}")
    
    if pd.notna(row['습관_improvement']):
        habit_details.append(f"습관 개선 상태: {row['습관_improvement']}")
    
    if habit_details:
        history_context.append("습관 상세정보: " + ", ".join(habit_details))
    
    # 병력 및 습관 섹션을 통합하여 전체 맥락에 추가
    if history_context:
        clinical_sections.append("【병력 및 습관】\n" + "\n".join(history_context))
    
    # 3. 치료 관련 섹션 (Treatment)
    treatment_context = []
    
    # 치료 계획
    if pd.notna(row['CC_treat_plan']):
        treatment_context.append(f"치료 계획: {row['CC_treat_plan']}")
    
    # 약물 관련 정보
    medication_details = []
    if pd.notna(row['약_medication_type']):
        medication_details.append(f"약물 유형: {row['약_medication_type']}")
    
    if pd.notna(row['약_frequency']):
        medication_details.append(f"복용 빈도: {row['약_frequency']}")
    
    if pd.notna(row['약_duration']):
        medication_details.append(f"복용 기간: {row['약_duration']}")
    
    if pd.notna(row['약_compliance']):
        medication_details.append(f"복약 순응도: {row['약_compliance']}")
    
    if medication_details:
        treatment_context.append("약물 정보: " + ", ".join(medication_details))
    
    # 장치 관련 정보
    device_details = []
    if pd.notna(row['장치_device_type']):
        device_details.append(f"장치 유형: {row['장치_device_type']}")
    
    if pd.notna(row['장치_usage_pattern']):
        device_details.append(f"사용 패턴: {row['장치_usage_pattern']}")
    
    if pd.notna(row['장치_duration']):
        device_details.append(f"사용 기간: {row['장치_duration']}")
    
    if pd.notna(row['장치_compliance']):
        device_details.append(f"장치 순응도: {row['장치_compliance']}")
    
    if device_details:
        treatment_context.append("장치 정보: " + ", ".join(device_details))
    
    # 찜질/마사지 정보
    therapy_details = []
    
    # 찜질 정보
    hot_pack_info = []
    if pd.notna(row['찜질_status']) and row['찜질_status'] == 1:
        hot_pack_info.append("찜질 시행")
        
        if pd.notna(row['찜질_frequency']):
            hot_pack_info.append(f"빈도: {row['찜질_frequency']}")
        
        if pd.notna(row['찜질_duration']):
            hot_pack_info.append(f"시간: {row['찜질_duration']}분")
        
        if pd.notna(row['찜질_method']):
            hot_pack_info.append(f"방법: {row['찜질_method']}")
    
    if hot_pack_info:
        therapy_details.append("찜질: " + ", ".join(hot_pack_info))
    
    # 마사지/스트레칭 정보
    massage_info = []
    if pd.notna(row['마사지, 스트레칭_type']):
        massage_info.append(f"유형: {row['마사지, 스트레칭_type']}")
        
        if pd.notna(row['마사지, 스트레칭_frequency']):
            massage_info.append(f"빈도: {row['마사지, 스트레칭_frequency']}")
        
        if pd.notna(row['마사지, 스트레칭_duration']):
            massage_info.append(f"시간: {row['마사지, 스트레칭_duration']}분")
        
        if pd.notna(row['마사지, 스트레칭_method']):
            massage_info.append(f"방법: {row['마사지, 스트레칭_method']}")
    
    if massage_info:
        therapy_details.append("마사지/스트레칭: " + ", ".join(massage_info))
    
    if therapy_details:
        treatment_context.append("치료 요법: " + "; ".join(therapy_details))
    
    # 치료 섹션을 통합하여 전체 맥락에 추가
    if treatment_context:
        clinical_sections.append("【치료 정보】\n" + "\n".join(treatment_context))
    
    # 4. 임상 측정 데이터 섹션 (Clinical Measurements)
    measurements_context = []
    
    # 입 벌림 관련 측정값
    opening_measurements = []
    if pd.notna(row['CMO_before']):
        opening_measurements.append(f"편안한 개구량(처음): {row['CMO_before']}mm")
    
    if pd.notna(row['CMO_after']):
        opening_measurements.append(f"편안한 개구량(이후): {row['CMO_after']}mm")
    
    if pd.notna(row['MMO_before']):
        opening_measurements.append(f"최대 개구량(처음): {row['MMO_before']}mm")
    
    if pd.notna(row['MMO_after']):
        opening_measurements.append(f"최대 개구량(이후): {row['MMO_after']}mm")
    
    if opening_measurements:
        measurements_context.append("개구량 측정: " + ", ".join(opening_measurements))
    
    # 턱 편위 관련 정보
    deviation_info = []
    if pd.notna(row['deviation_pattern_type']):
        deviation_info.append(f"편위 패턴: {row['deviation_pattern_type']}")
    
    if pd.notna(row['deviation_direction']):
        deviation_info.append(f"편위 방향: {row['deviation_direction']}")
    
    if pd.notna(row['deviation_intensity']):
        deviation_info.append(f"편위 강도: {row['deviation_intensity']}")
    
    if deviation_info:
        measurements_context.append("턱 편위: " + ", ".join(deviation_info))
    
    # 턱관절 소리 관련 정보
    noise_info = []
    if pd.notna(row['Noise_Code']) and row['Noise_Code'] != 'No-Noise':
        noise_info.append(f"소리 유형: {row['Noise_Code']}")
        
        if pd.notna(row['Noise_Direction']):
            noise_info.append(f"방향: {row['Noise_Direction']}")
        
        if pd.notna(row['Noise_Intensity']):
            noise_info.append(f"강도: {row['Noise_Intensity']}")
        
        if pd.notna(row['Noise_Situation']):
            noise_info.append(f"발생 상황: {row['Noise_Situation']}")
    
    if noise_info:
        measurements_context.append("턱관절 소리: " + ", ".join(noise_info))
    
    # 교합 관련 정보
    occlusion_info = []
    if pd.notna(row['Occlusion_lt_number']) or pd.notna(row['Occlusion_rt_number']):
        lt_info = f"왼쪽 교합 치아 수: {row['Occlusion_lt_number'] if pd.notna(row['Occlusion_lt_number']) else '정보 없음'}"
        rt_info = f"오른쪽 교합 치아 수: {row['Occlusion_rt_number'] if pd.notna(row['Occlusion_rt_number']) else '정보 없음'}"
        
        lt_intensity = f"왼쪽 교합 강도: {row['Occlusion_lt_Intensity'] if pd.notna(row['Occlusion_lt_Intensity']) else '정보 없음'}"
        rt_intensity = f"오른쪽 교합 강도: {row['Occlusion_rt_Intensity'] if pd.notna(row['Occlusion_rt_Intensity']) else '정보 없음'}"
        
        occlusion_info.extend([lt_info, rt_info, lt_intensity, rt_intensity])
    
    if pd.notna(row['oj']):
        occlusion_info.append(f"수평피개량: {row['oj']}mm")
    
    if pd.notna(row['ob']):
        occlusion_info.append(f"수직피개량: {row['ob']}mm")
    
    if occlusion_info:
        measurements_context.append("교합 상태: " + ", ".join(occlusion_info))
    
    # 임상 측정값 섹션을 통합하여 전체 맥락에 추가
    if measurements_context:
        clinical_sections.append("【임상 측정 데이터】\n" + "\n".join(measurements_context))
    
    # 5. 결과/치료 효과 섹션 (Outcomes)
    outcomes_context = []
    
    # 치료 효과 관련 정보
    if pd.notna(row['Next_Visit_Days']):
        outcomes_context.append(f"다음 방문 예정일: {row['Next_Visit_Days']}일 후")
    
    # VAS 점수 변화
    if pd.notna(row['CC_vas']):
        outcomes_context.append(f"현재 통증 점수(VAS): {row['CC_vas']}")
    
    # 결과 섹션을 통합하여 전체 맥락에 추가
    if outcomes_context:
        clinical_sections.append("【치료 결과】\n" + "\n".join(outcomes_context))
    
    # 모든 섹션을 통합하여 하나의 맥락화된 텍스트 생성
    contextual_text = "\n\n".join(clinical_sections)
    
    # 환자 ID와 날짜 정보를 헤더로 추가
    header = f"환자번호: {row['환자번호'] if pd.notna(row['환자번호']) else '정보 없음'}, 방문일: {row['날짜'] if pd.notna(row['날짜']) else '정보 없음'}"
    
    # 최종 맥락화된 텍스트
    final_text = f"{header}\n\n{contextual_text}"
    
    return final_text

def enhance_clinical_relationships(row):
    """임상적으로 관련된 필드들 간의 관계를 강화"""
    relationships = []
    
    # 증상과 치료의 관계
    if pd.notna(row['CC_pain_type']) and pd.notna(row['CC_treat_plan']):
        relationships.append(f"{row['CC_pain_type']}에 대한 치료 계획: {row['CC_treat_plan']}")
    
    # 습관과 증상의 관계
    if pd.notna(row['습관_habit_type']) and pd.notna(row['CC_muscle_joint_desc_stress']):
        relationships.append(f"습관 '{row['습관_habit_type']}'과 근육 상태 '{row['CC_muscle_joint_desc_stress']}'의 연관성")
    
    # 치료와 결과의 관계
    if pd.notna(row['장치_device_type']) and pd.notna(row['CC_vas']):
        relationships.append(f"{row['장치_device_type']} 장치 사용 후 통증 점수: {row['CC_vas']}")
    
    return relationships

def identify_patient_patterns(row):
    """환자 데이터에서 중요 패턴 식별"""
    patterns = []
    
    # 통증 위치와 종류 패턴
    if pd.notna(row['CC_location']) and pd.notna(row['CC_pain_type']):
        patterns.append(f"패턴-위치통증: {row['CC_location']}_{row['CC_pain_type']}")
    
    # 개구량 변화 패턴
    if pd.notna(row['MMO_before']) and pd.notna(row['MMO_after']):
        try:
            change = float(row['MMO_after']) - float(row['MMO_before'])
            if change > 3:
                patterns.append("패턴-개구량: 뚜렷한_개선")
            elif change > 0:
                patterns.append("패턴-개구량: 경미한_개선")
            elif change < -3:
                patterns.append("패턴-개구량: 악화")
            else:
                patterns.append("패턴-개구량: 유지")
        except:
            pass
    
    # 습관과 순응도 패턴
    if pd.notna(row['습관_awareness']) and pd.notna(row['장치_compliance']):
        patterns.append(f"패턴-순응도: 습관인지_{row['습관_awareness']}_장치순응도_{row['장치_compliance']}")
    
    return patterns

def apply_section_weights(text, section_weights):
    """섹션별 가중치를 적용한 텍스트 생성 - 특수 마커 사용"""
    sections = text.split("\n\n")
    weighted_sections = []
    
    for section in sections:
        for key, weight in section_weights.items():
            if key in section:
                # 섹션 시작 부분에 가중치 마커 추가
                weighted_section = f"<WEIGHT={weight}> {section}"
                weighted_sections.append(weighted_section)
                break
        else:
            weighted_sections.append(section)
    
    return "\n\n".join(weighted_sections)

def create_hybrid_contextual_embedding(df):
    """환자 데이터의 하이브리드 맥락 임베딩 생성"""
    
    # 임베딩 저장할 리스트
    embeddings = []
    
    for index, row in df.iterrows():
        # 1. 맥락화된 텍스트 생성
        row_dict = row.to_dict()
        clinical_text = create_contextual_text(row_dict)
        
        # 2. 임상적 관계 강화
        relationships = enhance_clinical_relationships(row_dict)
        if relationships:
            clinical_text += "\n\n【임상적 관계】\n" + "\n".join(relationships)
        
        # 3. 환자 패턴 식별 및 추가
        patterns = identify_patient_patterns(row_dict)
        if patterns:
            clinical_text += "\n\n【패턴 정보】\n" + "\n".join(patterns)
        
        # 4. 섹션 가중치 적용
        weighted_text = apply_section_weights(clinical_text, section_weights)
        
        # 맥락화된 텍스트 출력 (테스트용)
        print(f"환자 {row_dict['환자번호']}의 맥락화된 텍스트 샘플:\n{weighted_text[:500]}...\n")
        
        # 5. 임베딩 생성 (실제 OpenAI API 호출 대신 임시 벡터 생성)
        # 실제로는 아래 코드 대신 OpenAI API 호출 필요
        """
        response = openai.Embedding.create(
            model="text-embedding-ada-002",
            input=weighted_text
        )
        embedding = response['data'][0]['embedding']
        """
        # 임시 임베딩 생성 (실제 구현 시 주석 처리)
        temp_embedding = np.random.random(1536).tolist()  # OpenAI 임베딩 차원수 1536
        
        # 결과 저장
        embeddings.append({
            'patient_id': row_dict['환자번호'],
            'visit_date': row_dict['날짜'],
            'embedding': temp_embedding,
            'vas': row_dict['CC_vas'] if pd.notna(row_dict['CC_vas']) else None
        })
    
    return embeddings

# 사용 예시


In [47]:
tst = create_hybrid_contextual_embedding(df.head())

환자 2301-01의 맥락화된 텍스트 샘플:
환자번호: 2301-01, 방문일: 2023-01-17 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 오른쪽 턱에 경직을 호소합니다.
턱 기능 제한: 오른쪽 턱이 잘 안벌어짐

【임상 측정 데이터】
개구량 측정: 편안한 개구량(처음): 34mm, 최대 개구량(처음): 46mm
턱 편위: 편위 패턴: other, 편위 방향: unspecified, 편위 강도: normal
턱관절 소리: 소리 유형: Click, 방향: right, 강도: 0
교합 상태: 왼쪽 교합 치아 수: 0, 오른쪽 교합 치아 수: 0, 왼쪽 교합 강도: 0, 오른쪽 교합 강도: 0, 수평피개량: 2.0mm, 수직피개량: 2.0mm

【패턴 정보】
패턴-위치통증: 오른쪽 턱_경직...

환자 2301-01의 맥락화된 텍스트 샘플:
환자번호: 2301-01, 방문일: 2023-02-01 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 오른쪽 귀 앞에 불편함을 호소합니다.
턱 관련 통증 및 불편감: 턱이 불편함
근육 및 관절 상태: 치아 시린 부분

<WEIGHT=1.5> 【병력 및 습관】
습관 상세정보: 습관 유형: 치아 접촉 줄이기, 습관 인지 여부: 인지함

【치료 정보】
치료 요법: 찜질: 찜질 시행, 빈도: high, 시간: 10.0분, 방법: hot

【임상 측정 데이터】
개구량 측정: 편안한 개구량(처음): 38mm, 최대 개구량(처음): 46mm
턱 편위: 편위 패턴: other, 편위 방향: unspecified, 편위 강도: normal
턱관절 소리: 소리 유형: Click, 방향: right, 강도: 0
교합 상태: 왼쪽 교합 치아 수: 0, 오른쪽 교합 치아 수: 0, 왼쪽 교합 강도: 0, 오른쪽 교합 강도: 0, 수평피개량: 2.0mm, 수직피개량: 2.0mm

【임상...

환자 2301-01의 맥락화된 텍스트 샘플:
환자번호: 2301-01, 방문일: 2023-02-17 00:00:

In [49]:
pd.DataFrame(tst)

,patient_id,visit_date,embedding,vas
0,2301-01,2023-01-17,"[0.5490608028607417, 0.5545751551603862, 0.3359717420190622, 0.19313704300023726, 0.9378305686470809, 0.6867525310003278, 0.13164077290181075, 0.8392048884113419, 0.8417857926785498, 0.5444754860071152, 0.6105080618882492, 0.15762168468902782, 0.7276976562815601, 0.9333219810085962, 0.07313985832706305, 0.13210766009867625, 0.9867469379112791, 0.9599048466452883, 0.45959830018697, 0.25208619216750605, 0.030015834657593055, 0.5017228467726711, 0.028574371536945864, 0.5503772302569806, 0.30223375534616725, 0.0024181467386527045, 0.7354941867032752, 0.6941288787257012, 0.3670831356789568, 0.04211136237248403, 0.9442447890515154, 0.3149393552785962, 0.5728486650173793, 0.18946709395643835, 0.886075699657444, 0.3941171724739614, 0.2514962532665729, 0.027862513659815402, 0.797412928407077, 0.5182379557319189, 0.11589134187443084, 0.9251189484806043, 0.5853871135399343, 0.27641703973792153, 0.06899278601823688, 0.22980690122688163, 0.679532066993003, 0.1779064415364523, 0.9250710599485941, 0.87315641323523, 0.6896653553230794, 0.7026247827881206, 0.774285826018827, 0.3545013874668892, 0.5109058711382205, 0.6447396743852333, 0.5366087645912854, 0.0326212941174292, 0.6163640588986738, 0.9888003050898182, 0.805328630074075, 0.8224349249837714, 0.3025947617256717, 0.40277267100733116, 0.8064317498949007, 0.6145857624138398, 0.2382402328746095, 0.7275720429824438, 0.9659548007453866, 0.7306959530308609, 0.8660104376264063, 0.6443786079664101, 0.5156657151980891, 0.08649157987467293, 0.36108987926787095, 0.8727086709775396, 0.498110696028104, 0.6178500684934576, 0.8901462870924617, 0.16740210138690548, 0.6693271309124676, 0.07109497730759018, 0.6319927248106126, 0.69162029308748, 0.8739041218180111, 0.9223067180115395, 0.3161197169725888, 0.6613292845985553, 4.862773136171672e-05, 0.807482355422445, 0.27271693330796454, 0.931164235321209, 0.1670790431885809, 0.18124201337474688, 0.5712000376893451, 0.2506421313848449, 0.29128557574418135, 0.0503738802038417, 0.9997549384997302, 0.8077368430580554, ...]",None
1,2301-01,2023-02-01,"[0.01823005419748991, 0.03326658164161467, 0.9669119103725593, 0.6935178449183992, 0.6601439674896685, 0.045366410586121786, 0.29645222157474616, 0.9366303076828814, 0.3361274101977998, 0.32314111585457184, 0.3247833879555859, 0.5241305662042026, 0.9005551199369988, 0.3055881146884071, 0.6170036718851198, 0.30425807425392093, 0.5374882471451362, 0.2994266052759359, 0.9258346933592971, 0.1207754320219222, 0.059312151242514544, 0.780218302237379, 0.5393157306211164, 0.6145500705217227, 0.570967987259513, 0.9363558271729193, 0.7970776749850159, 0.23008191302305114, 0.7280735488394299, 0.5846975374651316, 0.7884351737450822, 0.44461992779119597, 0.9561974634979893, 0.872961135214216, 0.4832800444022127, 0.8885937554745418, 0.7277098296272961, 0.4836564786047254, 0.38815153692239346, 0.37860315567130276, 0.5043000403388992, 0.6444514124530297, 0.5098408717468201, 0.2367012981307145, 0.5631832441985472, 0.34659085734309925, 0.8532519590571126, 0.03508959749750595, 0.9318675653486952, 0.5101209840098578, 0.5323689456685167, 0.30186621549251347, 0.5240518876449255, 0.4100239791134018, 0.8171300344051007, 0.9163827219757217, 0.15671457416159928, 0.9592775584638343, 0.7288641611288249, 0.8619527310405589, 0.4248167049989031, 0.2767968857013522, 0.1313924994415997, 0.3554989739789097, 0.2821002976545328, 0.35522841955946016, 0.8818706232800028, 0.8089148921735854, 0.9195419450235927, 0.9108920692200406, 0.488112247309875, 0.07963100759134545, 0.7605481342982958, 0.16851812920941012, 0.5631826912720607, 0.07164578675785938, 0.5510681789043254, 0.2999665246657399, 0.7215896086703779, 0.6718949795886237, 0.40408270867864515, 0.36755203830710226, 0.6284566271831847, 0.19326171407066006, 0.40456160169927247, 0.550414211726603, 0.4413436853136822, 0.5260095967200491, 0.47459334844609946, 0.9287531435346406, 0.18702096561872084, 0.371329109025172, 0.10948187378230678, 0.414

In [ ]:
import openai
import numpy as np
import pandas as pd

# 텍스트 필드 선택
text_columns = ['CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw', 'CC_disable_desc_jaw', 
                'CC_muscle_joint_desc_stress', 'CC_factor_habbit', 'CC_treat_plan']

def create_embedding(row):
    # 누락되지 않은 텍스트 필드만 결합
    text = " ".join([str(row[col]) for col in text_columns if pd.notna(row[col])])
    if not text or text.isspace():
        return None
    
    # GPT 임베딩 생성
    response = openai.Embedding.create(
        model="text-embedding-ada-002",
        input=text
    )
    return response['data'][0]['embedding']

# 데이터프레임의 각 행에 임베딩 적용
df['text_embedding'] = df.apply(create_embedding, axis=1)